# Create Continuous RNA + Outer Join Datasets

Combines two fixes:
1. **Continuous RNA** from `star_counts.tsv` (z-scored) instead of discretized
2. **Outer join** — keeps all ~1076 patients, fills missing modalities with 0 + indicator

**Output:** `MERGE_continuous_outer/` — all previous folders untouched.

**Restart kernel before running.**

## Paths & Setup

In [3]:
"""
CREATE MERGED DATASETS WITH CONTINUOUS RNA VALUES

Problem: RNA in statistical_filtered/ was discretized (only 12 unique values
per gene) causing 388/526 patients to have duplicate RNA profiles.

Solution: Re-merge using raw continuous RNA from TCGA-BRCA.star_counts.tsv,
keeping the same MB-selected gene lists.

Output: MERGE_continuous_outer/  (original MERGE/ untouched)

Script location: 01_Causal_feature_extraction/
"""

import pandas as pd
import numpy as np
from pathlib import Path

# ============================================================================
# PATHS
# ============================================================================

try:
    SCRIPT_DIR = Path(__file__).resolve().parent
except NameError:
    cwd = Path.cwd()
    SCRIPT_DIR = cwd if cwd.name == "MB" else cwd

BASE_DIR    = SCRIPT_DIR
DATA_DIR    = BASE_DIR.parent / "data"
MERGE_DIR   = BASE_DIR / "MERGE"
OUT_DIR     = BASE_DIR / "MERGE_continuous_outer"
OUT_DIR.mkdir(exist_ok=True)

CLIN_FILE   = BASE_DIR / "Clinical" / "clinical_preprocessed_prefixed.csv"
RNA_RAW     = DATA_DIR / "TCGA-BRCA.star_counts.tsv"
MB_DIR      = BASE_DIR / "RNA" / "mb_results"

OUTCOME_CANDIDATES = [
    DATA_DIR / "outcome.csv",
    BASE_DIR / "RNA" / "preprocessed" / "outcome.csv",
]

print(f"Base dir:  {BASE_DIR}")
print(f"RNA raw:   {RNA_RAW.exists()} -> {RNA_RAW.name}")
print(f"Output:    {OUT_DIR}")

DATASET_NAMES = [
    "01_ultra_conservative", "02_conservative", "03_standard",
    "04_fdr_significant",    "05_balanced",     "06_correlation",
    "07_top_correlated",     "08_composite",
]

# MB result folder names matching each dataset (same order)
RNA_MB_DATASETS = [
    "rna_1_ultra_conservative_723genes",
    "rna_2_conservative_1079genes",
    "rna_3_standard_1751genes",
    "rna_4_fdr_significant_1000genes",
    "rna_5_balanced_1066genes",
    "rna_6_correlation_1081genes",
    "rna_7_top_correlated_200genes",
    "rna_8_composite_1000genes",
]

# Best configs per dataset (from mb_results summary — algorithm + alpha)
RNA_BEST_CONFIGS = [
    ("IAMB", 0.05),  # 01
    ("IAMB", 0.05),  # 02
    ("IAMB", 0.05),  # 03
    ("IAMB", 0.05),  # 04
    ("IAMB", 0.05),  # 05
    ("IAMB", 0.05),  # 06
    ("IAMB", 0.05),  # 07
    ("IAMB", 0.05),  # 08
]

MODALITY_PREFIXES = {
    "CNV":   "CNV_",
    "MUT":   "MUT_",
    "PROT":  "PROT_",
    "METH":  "METH_",
    "MIRNA": "MIRNA_",
}

Base dir:  C:\Users\olegk\Desktop\Thesis_v3\01_Causal_feature_extraction
RNA raw:   True -> TCGA-BRCA.star_counts.tsv
Output:    C:\Users\olegk\Desktop\Thesis_v3\01_Causal_feature_extraction\MERGE_continuous_outer


## Step 1 — Load Clinical & Outcome

In [5]:
# ============================================================================
# STEP 1: LOAD CLINICAL + OUTCOME
# ============================================================================
print("\n" + "="*70)
print("STEP 1: LOAD CLINICAL + OUTCOME")
print("="*70)

clinical = pd.read_csv(CLIN_FILE, index_col=0)
print(f"Clinical: {clinical.shape}")

outcome = None
for cand in OUTCOME_CANDIDATES:
    if cand.exists():
        outcome = pd.read_csv(cand, index_col=0)
        outcome.index = [
            "-".join(str(i).replace(".", "-").split("-")[:3])
            for i in outcome.index
        ]
        print(f"Outcome:  {outcome.shape}  (from {cand.name})")
        break

if outcome is None:
    raise FileNotFoundError("outcome.csv not found")


STEP 1: LOAD CLINICAL + OUTCOME
Clinical: (1098, 144)
Outcome:  (1076, 2)  (from outcome.csv)


## Step 2 — Load & Preprocess Raw RNA

In [7]:
# ============================================================================
# STEP 2: LOAD + PREPROCESS RAW RNA
# ============================================================================
print("\n" + "="*70)
print("STEP 2: LOAD RAW RNA (star_counts.tsv)")
print("="*70)

rna_raw = pd.read_csv(RNA_RAW, sep="\t", index_col=0)
print(f"Raw RNA shape (genes x samples): {rna_raw.shape}")

# Transpose: samples as rows, genes as columns
rna_raw = rna_raw.T
print(f"Transposed (samples x genes):    {rna_raw.shape}")

# Normalize sample IDs to patient level (TCGA-XX-XXXX)
def normalize_id(idx):
    parts = str(idx).replace(".", "-").split("-")
    return "-".join(parts[:3]) if len(parts) >= 3 else str(idx)

rna_raw.index = [normalize_id(i) for i in rna_raw.index]

# Filter to Primary Tumor only (original barcodes end with -01X)
# Already transposed so we work with original index before normalization
# Re-load to filter properly
rna_full = pd.read_csv(RNA_RAW, sep="\t", index_col=0)
tumor_cols = [c for c in rna_full.columns if "-01" in c]
rna_tumor  = rna_full[tumor_cols].T
rna_tumor.index = [normalize_id(i) for i in rna_tumor.index]

# Average duplicates (same patient, multiple tumor samples)
if rna_tumor.index.duplicated().any():
    n_dup = rna_tumor.index.duplicated(keep=False).sum()
    print(f"Averaging {n_dup} duplicate tumor samples...")
    rna_tumor = rna_tumor.groupby(rna_tumor.index).mean()

print(f"After Primary Tumor filter + dedup: {rna_tumor.shape}")

# Z-score normalize per gene
rna_zscore = (rna_tumor - rna_tumor.mean()) / (rna_tumor.std() + 1e-8)
print(f"Z-scored RNA: {rna_zscore.shape}")
print(f"Unique values check (should be >> 12): "
      f"{rna_zscore.iloc[:, 0].nunique()} for first gene")


STEP 2: LOAD RAW RNA (star_counts.tsv)
Raw RNA shape (genes x samples): (60660, 1226)
Transposed (samples x genes):    (1226, 60660)
Averaging 22 duplicate tumor samples...
After Primary Tumor filter + dedup: (1095, 60660)
Z-scored RNA: (1095, 60660)
Unique values check (should be >> 12): 998 for first gene


## Step 3 — Build Merged Datasets

In [9]:
# ============================================================================
# STEP 3: BUILD MERGED DATASETS
# ============================================================================
print("\n" + "="*70)
print("STEP 3: BUILD CONTINUOUS MERGED DATASETS")
print("="*70)

summary = []

for i, ds_name in enumerate(DATASET_NAMES):
    inner_file = MERGE_DIR / f"{ds_name}.csv"
    mb_dataset = RNA_MB_DATASETS[i]
    algo, alpha = RNA_BEST_CONFIGS[i]

    print(f"\n{'─'*70}")
    print(f"[{ds_name}]")

    # ---- Load MB gene list ----
    for fmt in [f"{alpha:.2f}", str(alpha)]:
        gene_file = MB_DIR / mb_dataset / f"{algo}_alpha{fmt}_genes.txt"
        if gene_file.exists():
            break
    if not gene_file.exists():
        print(f"  Gene file not found: {gene_file} — skipping")
        continue

    genes = [l.strip() for l in gene_file.read_text().splitlines() if l.strip()]
    # Strip version suffix for matching (ENSG00000228098.1 -> try both)
    available_genes = [g for g in genes if g in rna_zscore.columns]
    if not available_genes:
        # Try stripping version
        gene_map = {g.split(".")[0]: g for g in rna_zscore.columns}
        available_genes = []
        for g in genes:
            base = g.split(".")[0]
            if base in gene_map:
                available_genes.append(gene_map[base])
    print(f"  MB genes: {len(genes)} selected, {len(available_genes)} found in RNA")

    # ---- Extract RNA features ----
    rna_sel = rna_zscore[available_genes].copy()
    rna_sel.columns = [f"RNA_{c}" for c in rna_sel.columns]

    # ---- Start merge: clinical (base = all patients) ----
    merged = clinical.copy()

    # ---- Add RNA (outer join + missingness indicator) ----
    merged = merged.join(rna_sel, how="left")
    rna_missing = merged[rna_sel.columns].isnull().all(axis=1).astype(int)
    merged["RNA_missing"] = rna_missing
    merged[rna_sel.columns] = merged[rna_sel.columns].fillna(0)
    n_with_rna = (rna_missing == 0).sum()
    print(f"  RNA: {n_with_rna} with data, {rna_missing.sum()} missing → indicator added")

    # ---- Add other modalities (outer join + missingness indicators) ----
    if inner_file.exists():
        inner = pd.read_csv(inner_file, index_col=0)
        for mod_name, prefix in MODALITY_PREFIXES.items():
            mod_cols = [c for c in inner.columns if c.startswith(prefix)]
            if mod_cols:
                mod_data = inner[mod_cols]
                merged = merged.join(mod_data, how="left")
                mod_missing = merged[mod_cols].isnull().all(axis=1).astype(int)
                merged[f"{prefix}missing"] = mod_missing
                merged[mod_cols] = merged[mod_cols].fillna(0)
                n_with = (mod_missing == 0).sum()
                print(f"  {mod_name}: {n_with} with data, {mod_missing.sum()} missing → indicator added")
    else:
        print(f"  WARNING: inner file not found, skipping other modalities")

    # ---- Add outcome (inner join — need OS/OS.time) ----
    merged = merged.join(outcome[["OS", "OS.time"]], how="inner")

    # ---- Verify uniqueness ----
    rna_cols_out = [c for c in merged.columns if c.startswith("RNA_")]
    n_unique_rna = merged[rna_cols_out].drop_duplicates().shape[0]
    print(f"  Unique RNA profiles: {n_unique_rna} of {len(merged)}")

    n_events = int(merged["OS"].sum())
    print(f"  Final shape: {merged.shape}  |  events: {n_events}")

    merged.to_csv(OUT_DIR / f"{ds_name}.csv")
    print(f"  Saved: {ds_name}.csv")

    summary.append({
        "dataset":        ds_name,
        "n_samples":      len(merged),
        "n_events":       n_events,
        "n_features":     merged.shape[1] - 2,
        "n_rna_genes":    len(rna_cols_out),
        "unique_rna_pct": round(n_unique_rna / len(merged) * 100, 1),
    })


STEP 3: BUILD CONTINUOUS MERGED DATASETS

──────────────────────────────────────────────────────────────────────
[01_ultra_conservative]
  MB genes: 50 selected, 50 found in RNA
  RNA: 1095 with data, 3 missing → indicator added
  CNV: 526 with data, 572 missing → indicator added
  MUT: 526 with data, 572 missing → indicator added
  PROT: 526 with data, 572 missing → indicator added
  METH: 526 with data, 572 missing → indicator added
  MIRNA: 526 with data, 572 missing → indicator added
  Unique RNA profiles: 318 of 1076
  Final shape: (1076, 452)  |  events: 150
  Saved: 01_ultra_conservative.csv

──────────────────────────────────────────────────────────────────────
[02_conservative]
  MB genes: 50 selected, 50 found in RNA
  RNA: 1095 with data, 3 missing → indicator added
  CNV: 526 with data, 572 missing → indicator added
  MUT: 526 with data, 572 missing → indicator added
  PROT: 526 with data, 572 missing → indicator added
  METH: 526 with data, 572 missing → indicator added
 

## Step 4 — Summary

In [11]:
# ============================================================================
# STEP 4: SUMMARY
# ============================================================================
print("\n" + "="*70)
print("STEP 4: SUMMARY")
print("="*70)

if summary:
    df_sum = pd.DataFrame(summary)
    print(df_sum.to_string(index=False))
    df_sum.to_csv(OUT_DIR / "continuous_outer_summary.csv", index=False)
    print(f"\nSaved to: {OUT_DIR}")
    print(f"Original MERGE/ untouched.")


STEP 4: SUMMARY
              dataset  n_samples  n_events  n_features  n_rna_genes  unique_rna_pct
01_ultra_conservative       1076       150         450           51            29.6
      02_conservative       1076       150         450           51            49.5
          03_standard       1076       150         450           51            80.4
   04_fdr_significant       1076       150         450           51            99.8
          05_balanced       1076       150         450           51            99.8
       06_correlation       1076       150         450           51            99.8
    07_top_correlated       1076       150         450           51            99.8
         08_composite       1076       150         450           51            99.8

Saved to: C:\Users\olegk\Desktop\Thesis_v3\01_Causal_feature_extraction\MERGE_continuous_outer
Original MERGE/ untouched.
